# (re-)Run the matchers

This does the rubin and desc meta-dectection first, then does a meta-matching

In [ ]:
# Standard imports
import tables_io
import hpmcm
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Set the tract
tract = 8006
# This will need to change with tract
ref_dir=(340.5000000000, -9.6694214876),
our_geom = hpmcm.shear_utils.ShearCellGeometry(ref_dir=ref_dir)

In [ ]:
# If this is True only run on a single cell
test_only = False

In [ ]:
# Files and catalogs ids for rubin metadetect
rubin_shear_match_files = [
    f'{tract}/shear_object_reduced_uncleaned_{tract}_ns.parq',
    f'{tract}/shear_object_reduced_uncleaned_{tract}_1m.parq',
    f'{tract}/shear_object_reduced_uncleaned_{tract}_1p.parq',
    f'{tract}/shear_object_reduced_uncleaned_{tract}_2m.parq',
    f'{tract}/shear_object_reduced_uncleaned_{tract}_2p.parq',    
]
rubin_shear_match_ids = list(range(len(rubin_shear_match_files)))

# Files and catalogs ids for desc metadetect
desc_shear_match_files = [
    f'{tract}/mdet-run-reduced_uncleaned_{tract}_ns.parq',
    f'{tract}/mdet-run-reduced_uncleaned_{tract}_1m.parq',
    f'{tract}/mdet-run-reduced_uncleaned_{tract}_1p.parq',
]
desc_shear_match_ids = list(range(len(desc_shear_match_files)))

# Files and catalogs ids for meta-matching
all_files = [
    f'{tract}/object_reduced_{tract}.parq',
    f'{tract}/anacal_reduced_{tract}.parq',
    f'{tract}/desc_md_match_object_stats.pq',
    f'{tract}/rubin_md_match_object_stats.pq',
]
all_ids = list(range(len(all_files)))

In [ ]:
# Allow for re-running various matchers
rematch_rubin_shear = True
rematch_desc_shear = True
rematch_objects = True

if rematch_rubin_shear:
    rubin_shear_match = hpmcm.shear_match.ShearMatch.createShearMatch(
        geometry=our_geom,
        deshear=-0.01,
        shear_names=['ns', '1m', '1p', '2m', '2p'],        
    )
    rubin_shear_match.reduceData(rubin_shear_match_files, rubin_shear_match_ids)
    if test_only:
        rubin_shear_match.analysisLoop(range(100, 101), range(100, 101))
    else:
        rubin_shear_match.analysisLoop()
    rubin_stats = rubin_shear_match.extractStats()
    rubin_stats.update(**rubin_shear_match.extractShearStats())
    tables_io.write(rubin_stats, f'{tract}/rubin_md_match_')
    rubin_shear_match.save(f'{tract}/rubin_md_match')


if rematch_desc_shear:
    desc_shear_match = hpmcm.shear_match.ShearMatch.createShearMatch(
        geometry=our_geom,
        deshear=-0.01,
        shear_names=['ns', '1m', '1p'],
    )
    desc_shear_match.reduceData(desc_shear_match_files, desc_shear_match_ids)
    if test_only:    
        desc_shear_match.analysisLoop(range(100, 101), range(100, 101))
    else:
        desc_shear_match.analysisLoop()
    desc_stats = desc_shear_match.extractStats()
    desc_stats.update(**desc_shear_match.extractShearStats())
    tables_io.write(desc_stats, f'{tract}/desc_md_match_')    
    desc_shear_match.save(f'{tract}/desc_md_match')


if rematch_objects:
    pixel_scale = 0.2
    n_pix_region = 30000
    region_size_deg = n_pix_region*pixel_scale/3600
    wcs_match = hpmcm.WcsMatch.create(
        ref_dir=ref_dir,
        region_size=(region_size_deg, region_size_deg),
        pixel_size=pixel_scale/3600,
        cell_size=150,
        cell_buffer=5,
        cell_max_object=10000,
    )
    
    wcs_match.reduceData(all_files, all_ids)
    if test_only:
        wcs_match.analysisLoop(range(100, 101), range(100, 101))
    else:
        wcs_match.analysisLoop()
    stats = wcs_match.extractStats()
    tables_io.write(stats, f'{tract}/all_match_')    
    wcs_match.save(f'{tract}/all_match')

In [ ]:
# Draw a cluster from the rubin shear matching
cell_data = rubin_shear_match.cell_dict[20200]
cluster_id = list(cell_data.cluster_dict.keys())[1]
cluster_data = cell_data.cluster_dict[cluster_id]
img = cell_data.countsMap()
_ = hpmcm.viz_utils.showCluster(
    img,
    cluster_data,
    cell_data,
)

In [ ]:
# Draw a cluster from the desc shear matching
cell_data = desc_shear_match.cell_dict[20200]
cluster_id = list(cell_data.cluster_dict.keys())[1]
cluster_data = cell_data.cluster_dict[cluster_id]
img = cell_data.countsMap()
_ = hpmcm.viz_utils.showCluster(
    img,
    cluster_data,
    cell_data,
)

In [ ]:
# Draw a cluster from the meta matching
cell_data = wcs_match.cell_dict[20200]
cluster_id = list(cell_data.cluster_dict.keys())[0]
cluster_data = cell_data.cluster_dict[cluster_id]
img = cell_data.countsMap()
_ = hpmcm.viz_utils.showCluster(
    img,
    cluster_data,
    cell_data,
)